# 03 · Verifier calibration  _(owner: M3)_

Score the Verifier against MedAESQA human labels; iterate the prompt until Cohen's κ ≥ 0.6 / agreement ≥ 80%; show confusion matrices.

In [26]:
import json
import pandas as pd
import numpy as np
import textwrap
import random


plaba = json.load(open("../data/plaba/data.json"))

### Some requirements if a response is faithful or not

Divide the draft into individual clinical claims. Split sentences containing multiple claims.

Compare each claim with the source.

Check especially for changes to:

- Negation: “no evidence of” versus “has.”

- Certainty: “possible” versus “confirmed.”

- Numbers, units, doses, frequency, duration, and dates.

- Who the statement concerns.

- Past, current, planned, and conditional events.

- Associations versus causes.

We can possibility split the draft and source, and label them to a specific claim and see if the draft states the same information as the source.



### Example
Source: “Possible infection. Follow up in two weeks.”

- Draft: “You may have an infection. Return in two weeks.”
    - Judgment: Both claims are supported; no essential omission.

- Draft: “You definitely have an infection. Return in two weeks.”
    - Judgment: The first claim is unsupported because the source establishes only a possibility.

- Draft: “You may have an infection. Return in two days.”
    - Judgment: The follow-up timing is contradicted by the source.

- Draft: “You may have an infection.”
    - Judgment: The claim is supported, but the follow-up instruction is an essential omission.

# Rubric

### Human Grader

Decide whether an AI-output is Faithful or not.

Faithful means 
- if all claims from the output supports the source.
- contradicts nothing in the source, if it does then it is unfaithful
- there is none omission in the draft.


### Step 1. Split the draft into different claims

Break each sentence into single claims
- Example: Cramps are common and usually harmless -> Cramps are common, they are usually harmless

### Step 2. Label each draft claim

We label each claim with either **Supported**, or **Unsupported**

- **Supported**: The source states or clearly implies the draft
- **Unsupported**: The source doesn't support the draft, it added additional fact or either invented new information.
    - added facts or invented details,
    - claims the source contradicts,
    - claims that are stronger, weaker or more specific than the source.

#### Each Supported claim should check for:
- Negation: "no evidence of" vs. "has"
- Certainty: "possible/may" vs. "confirmed/definitely"
- Numbers: amounts, units, doses, frequency, duration, dates
- Subject: who the statement is about
- Time: past, current, planned, conditional
- Cause vs. association: "linked to" vs. "causes"

### Step 3. Check for Omissions

Read the source and list any claims the draft doesn't cover, We want the omission to only be wordings that arent required in the simplified prompt, so we can label if it should be omitted or not.

### Step 6. Verdict
faithful = true only if unsupported_claims is empty and omissions is empty. Otherwise faithful = false. One problem of either kind is enough to fail.



### Worked examples
Source: "Possible infection. Follow up in two weeks."

| Draft | unsupported_claims | omissions | faithful |
|---|---|---|---|
| "You may have an infection. Return in two weeks." | [] | [] | true |
| "You definitely have an infection. Return in two weeks." | ["You definitely have an infection" (certainty)] | [] | false |
| "You may have an infection. Return in two days." | ["Return in two days" (duration; source says two weeks)] | [] | false |
| "You may have an infection." | [] | ["Follow up in two weeks"] | false |

Flag the categories for the unsupported claims



### Edge cases
- **Extra explanation** (defining a term, giving advice): Unsupported unless the source says it.
    - The readability will reword the source term, if there isn't a definition in the source, dont include it. Don't include extra claims in the draft.
- **Vague wording** ("some people" for "10% of patients"): Unsupported. It changes the number.
    - Example: "10% of patients" -> "some people"
- **Ambiguous source:** use the most literal reading, and add a note.
    - Example: "Take with food or after meals" could mean either one is fine, or both are required.
- **Reordered or merged sentences:** Reorder the claims the same way you found it, do not change claims or claim ordering.
- **Repeated claims:** list a problem once.




In [ ]:
from __future__ import annotations

import sys
sys.path.append("../src")  # notebooks/ sits next to src/

from clearmed.models import DEFAULT_MODEL, call_model

PROMPT = """You are a medical faithfulness verifier. You compare a DRAFT (a plain-language rewrite) against the SOURCE facts (ATOMS). Use only the SOURCE. Do not use outside medical knowledge: a claim that is true but absent from the SOURCE is unsupported.

For your task, you will only perform checking and labeling if the Draft is a faithful rewrite of the SOURCE. If it is not, you will return a list of unsupported claims.

Follow these steps exactly.

STEP 1 - Split BOTH The DRAFT and SOURCE first into sentences, then from the senten atomic claims. An atomic claim is ONE fact that could be true or false on its own. Never treat a whole sentence as one claim. Split on:
- every adjective or modifier ("a temporary but intense and painful contraction" = 3 claims: temporary, intense, painful)
- every "and", "but", "although", "while", and semicolon
- every "that ..." or "which ..." clause
- every item in a list
Write each claim as a short full sentence with its subject, numbered in order.
Example: "Muscle cramp is a temporary but intense involuntary contraction that can occur in many situations."
S1 Muscle cramp is temporary. S2 Muscle cramp is intense. S3 Muscle cramp is involuntary. S4 Muscle cramp can occur in many situations.
If a sentence has commas, "and", "but" or "that" and you produced only one claim for it, split it again.

STEP 2 - Match and label each DRAFT claim.

For each DRAFT claim, find the SOURCE claim that best matches it. Label the DRAFT claim as either Supported or Unsupported.
- Supported: the SOURCE states it or clearly implies it. Rewording is fine if it means the same thing to a lay reader (involuntary = uncontrollable, laboratory = lab, likelihood = chance).
- Unsupported: everything else. This includes added facts, invented details, claims the SOURCE contradicts, claims that are stronger, weaker, or more specific than the SOURCE, and claims whose meaning changed even though the words look similar (for example "situations" became "reasons"; "frequency" became "amount"). If you would have to guess or use outside knowledge to call a claim Supported, it is Unsupported.

STEP 3 - Change check. A claim is Unsupported if it differs from the SOURCE on any of:

- Negation ("no evidence of" vs. "has") Cannot be contridicted by the SOURCE.
- Certainty ("seems likely/appears/may" vs. "is/definitely"; "unknown" vs. "uncertain") Don't assume certainty, if the source states "unknown" or "uncertain", the draft cannot say "definitely" or "is".
- Numbers, units, doses, frequency, duration, dates(DO NOT FAKE NUMBERS OR DATES OR UNITS)
- Who or what the statement concerns
- Time (past, current, planned, conditional; "during or soon after exercise" vs. "exercise-related")
- Cause vs. association ("causes", "involves", "linked to")
Also treat extra explanation or advice not in the SOURCE, and vaguer wording that loses a number ("some people" for "10%"), as Unsupported.


Rules:
- Match claims by meaning, not by position. Reordering or merging sentences is fine as long as every SOURCE claim is still covered and no claim changed.
- List each problem once. Quote the exact draft claim in "unsupported_claims".
-Return the unsupported claims in a JSON object with the key "unsupported_claims" and a list of strings as the value. If there are no unsupported claims, return an empty list.
Output format:
{"unsupported_claims": [str]}
"""



def verify(draft: str, atoms: str, model: str = DEFAULT_MODEL) -> str:
    """Return a JSON verdict string comparing `draft` to `atoms`."""
    return call_model(PROMPT, f"DRAFT:\n{draft}\n\nATOMS:\n{atoms}", model=model)  # dont change, for free / local ollama models to avoid api keys


In [36]:

entries = {pmid: entry
           for group in plaba.values()
           for pmid, entry in group.items()
           if isinstance(entry, dict)}  # skips "question" and "question_type"

random_pmids = random.Random(42).sample(list(entries), 10)
print(random_pmids)


for pmid in random_pmids:
    entry = entries[pmid]
    abstract = textwrap.fill(join(entry["abstract"]), width=90)

    for name, sentences in entry["adaptations"].items():
        print(abstract + "\n")
        text = textwrap.fill(join(sentences), width=90)
        print(text + "\n")
        result = verify(draft=text, atoms=abstract)
        print(f"{pmid} {name}:\n{result}\n")


['34775776', '18589845', '29244647', '32784601', '32150748', '32668217', '27306796', '31161145', '23334322', '32270976']
Chylothorax is an uncommon and serious clinical condition, typically induced by trauma,
either postsurgical or accidental injury, but the mechanism of chylothorax caused by
nephrotic syndrome is still unclear. Here, we report a case of primary nephrotic syndrome
with membranous nephropathy (MN) in a 66-year-old man who presented with severe
chylothorax. The chylothorax was managed by intercostal chest tube drainage, subcutaneous
injection of enoxaparin, and treatment with anti-inflammatory agents and diuretics. After
treatment, the patient's pleural effusion decreased, and the chyle gradually became clear.
We discuss the causes of MN with chylothorax. We considered that the hypoproteinemia
changed the permeability of mucous membranes and lymphatic vessels, leading to leakage of
chylous particles and chylous pleural effusion formation. Chylothorax may also have been
c